In [110]:
# load libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from lifelines.utils import concordance_index

import pyhere as here
import random
import copy

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Data Loading and Initial Processing

In [111]:
# ============================================================
# CLINICAL + MUTATION FEATURE BUILDER
# ============================================================

class ClinicalMutationBuilder:
    """
    Builds X_clin from:
      - clinical metadata
      - mutation features (binary + severity + TMB)
    """

    def __init__(self):
        self.cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.num_imputer = SimpleImputer(strategy="median")

        self.fitted = False
        self.feature_names = None

    # -----------------------------
    # FIT
    # -----------------------------
    def fit(self, clinical_df, mutation_df):
        """
        Fit encoders ONLY on training data (CV-safe)
        """

        clinical_df = clinical_df.copy()
        mutation_df = mutation_df.copy()

        # -----------------------------
        # 1. Identify feature types
        # -----------------------------
        cat_cols = clinical_df.select_dtypes(include=["object", "category"]).columns
        num_cols = clinical_df.select_dtypes(include=["int64", "float64"]).columns

        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)

        # -----------------------------
        # 2. Fit encoders
        # -----------------------------
        if len(self.cat_cols) > 0:
            self.cat_encoder.fit(clinical_df[self.cat_cols])

        if len(self.num_cols) > 0:
            self.num_imputer.fit(clinical_df[self.num_cols])

        # -----------------------------
        # 3. Mutation schema sanity check
        # -----------------------------
        expected_mut_cols = mutation_df.shape[1]
        self.mutation_dim = expected_mut_cols

        self.fitted = True
        return self

    # -----------------------------
    # TRANSFORM
    # -----------------------------
    def transform(self, clinical_df, mutation_df):
        assert self.fitted, "Must call fit() first"

        clinical_df = clinical_df.copy()
        mutation_df = mutation_df.copy()

        # -----------------------------
        # 1. Clinical numeric
        # -----------------------------
        if len(self.num_cols) > 0:
            clin_num = self.num_imputer.transform(clinical_df[self.num_cols])
        else:
            clin_num = np.zeros((len(clinical_df), 0))

        # -----------------------------
        # 2. Clinical categorical
        # -----------------------------
        if len(self.cat_cols) > 0:
            clin_cat = self.cat_encoder.transform(clinical_df[self.cat_cols])
        else:
            clin_cat = np.zeros((len(clinical_df), 0))

        # -----------------------------
        # 3. Mutation handling
        # -----------------------------
        mut = mutation_df.values.astype(np.float32)

        # Replace NaNs / infs in mutation data
        mut = np.nan_to_num(mut, nan=0.0, posinf=0.0, neginf=0.0)

        # IMPORTANT: do NOT standardize mutation features
        # (keeps binary sparsity intact)

        # -----------------------------
        # 4. Concatenate all clinical + mutation
        # -----------------------------
        X_clin = np.hstack([clin_num, clin_cat, mut])

        # -----------------------------
        # 5. Final safety check
        # -----------------------------
        X_clin = np.nan_to_num(X_clin, nan=0.0, posinf=0.0, neginf=0.0)

        return X_clin


In [112]:
# ============================================================
# GLOBAL FEATURE CLEANER (RECOMMENDED BEFORE CV)
# ============================================================

def remove_constant_features(X, threshold=1e-8):
    """
    Removes zero-variance / near-zero variance features.
    CRITICAL for preventing StandardScaler NaNs.
    """

    var = X.var(axis=0)
    keep = var > threshold

    X_clean = X[:, keep]

    print(f"[Feature Filter]")
    print(f"Original: {X.shape[1]} features")
    print(f"Kept: {X_clean.shape[1]} features")
    print(f"Dropped: {np.sum(~keep)} constant features")

    return X_clean, keep

In [113]:
# ============================================================
# OPTIONAL: SAFE MUTATION BUILDER (if you have raw mutation tables)
# ============================================================

def build_mutation_matrix(mut_binary_df, mut_severity_df=None, tmb_series=None):
    """
    Constructs mutation feature matrix safely.
    """

    mut_binary = mut_binary_df.values.astype(np.float32)
    mut_binary = np.nan_to_num(mut_binary)
    
    features = [mut_binary]

    if mut_severity_df is not None:
        mut_sev = mut_severity_df.values.astype(np.float32)
        mut_sev = np.log1p(np.nan_to_num(mut_sev))
        features.append(mut_sev)

    if tmb_series is not None:
        tmb = np.log1p(np.nan_to_num(tmb_series.values.astype(np.float32))).reshape(-1, 1)
        features.append(tmb)

    X_mut = np.hstack(features)

    return X_mut

In [114]:
def create_X_clin_pipeline(clinical_df, mutation_df, y_event):
    """
    Full safe pipeline to create X_clin
    """

    # -----------------------------
    # 1. Build raw matrix
    # -----------------------------
    builder = ClinicalMutationBuilder()

    builder.fit(clinical_df, mutation_df)

    X_clin = builder.transform(clinical_df, mutation_df)

    # -----------------------------
    # 2. Remove constant features globally
    # -----------------------------
    X_clin, keep_mask = remove_constant_features(X_clin)

    # -----------------------------
    # 3. Final safety cleanup
    # -----------------------------
    X_clin = np.nan_to_num(X_clin, nan=0.0, posinf=0.0, neginf=0.0)

    print("\nFinal X_clin shape:", X_clin.shape)

    return X_clin, builder, keep_mask

In [115]:
# ============================================================
# COX LOSS (STABLE)
# ============================================================
def cox_loss(risk_scores, times, events):
    order = torch.argsort(times, descending=True)

    risk_scores = risk_scores[order]
    events = events[order]

    log_cumsum = torch.logcumsumexp(risk_scores, dim=0)

    loss = -torch.sum((risk_scores - log_cumsum) * events)

    n_events = events.sum().clamp_min(1.0)

    return loss / n_events

In [138]:
from lifelines import CoxPHFitter
import pandas as pd
import numpy as np

# ============================================================
# Univariate Cox Gene Selector
# ============================================================

def select_survival_genes(X_expr, y_time, y_event,
                          top_k=100,
                          min_variance=1e-8):

    if isinstance(X_expr, pd.DataFrame):
        gene_names = X_expr.columns.to_numpy()
        X_expr = X_expr.to_numpy()
    else:
        gene_names = np.arange(X_expr.shape[1])

    y_time = np.asarray(y_time)
    y_event = np.asarray(y_event)

    n_genes = X_expr.shape[1]
    pvals = np.ones(n_genes)

    for i in range(n_genes):

        gene = X_expr[:, i]

        if np.var(gene) < min_variance:
            continue

        df = pd.DataFrame({
            "time": y_time,
            "event": y_event,
            "gene": gene
        })

        try:
            cph = CoxPHFitter(penalizer=0.01)
            cph.fit(df,
                    duration_col="time",
                    event_col="event")

            pvals[i] = cph.summary.loc["gene", "p"]

        except Exception:
            pvals[i] = 1.0

    selected_idx = np.argsort(pvals)[:top_k]
    selected_genes = gene_names[selected_idx]

    return selected_idx, selected_genes

In [167]:
# ============================================================
# Faster Gene Selector Function
# ============================================================
def select_survival_genes_fast(
    X_expr,
    y_time,
    y_event,
    top_k=100
):

    if isinstance(X_expr, pd.DataFrame):
        gene_names = X_expr.columns.to_numpy()
        X_expr = X_expr.to_numpy()
    else:
        gene_names = np.arange(X_expr.shape[1])

    X = np.asarray(X_expr, dtype=np.float64)
    times = np.asarray(y_time)
    events = np.asarray(y_event)

    # Sort by time
    order = np.argsort(times)

    X = X[order]
    events = events[order]

    # Risk set means
    rev_cumsum = np.cumsum(X[::-1], axis=0)[::-1]

    risk_counts = np.arange(len(X), 0, -1)

    risk_means = rev_cumsum / risk_counts[:, None]

    # Score statistics
    event_mask = events == 1

    U = np.sum(
        X[event_mask] - risk_means[event_mask],
        axis=0
    )

    scores = np.abs(U)

    selected_idx = np.argsort(scores)[::-1][:top_k]

    selected_genes = gene_names[selected_idx]

    return selected_idx, selected_genes, scores[selected_idx]

In [139]:
# ============================================================
# PREPROCESSOR (LEAKAGE SAFE)
# ============================================================
class SurvivalPreprocessor:
    def __init__(self):
        self.scaler_clin = StandardScaler()
        self.scaler_expr = StandardScaler()
        self.is_fitted = False

    def fit(self, X_clin, X_expr):
        self.scaler_clin.fit(X_clin)
        self.scaler_expr.fit(X_expr)
        self.is_fitted = True

    def transform(self, X_clin, X_expr):
        assert self.is_fitted

        X_clin = self.scaler_clin.transform(X_clin)
        X_expr = self.scaler_expr.transform(X_expr)

        X_clin = np.nan_to_num(X_clin, nan=0.0, posinf=0.0, neginf=0.0)
        X_expr = np.nan_to_num(X_expr, nan=0.0, posinf=0.0, neginf=0.0)

        return X_clin, X_expr

In [140]:
# ============================================================
# SURVIVAL AWARE AUTOENCODER
# ============================================================
class SurvivalAwareAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

        self.survival_head = nn.Linear(latent_dim, 1)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        risk = self.survival_head(z).squeeze(-1)
        return x_recon, z, risk

In [141]:
# ============================================================
# DUAL ENCODER SURVIVAL MODEL
# ============================================================
class DualEncoderSurvivalNet(nn.Module):
    def __init__(self, clin_mut_dim, expr_autoencoder,
                 latent_dim_clin=32, latent_dim_expr=64):

        super().__init__()

        self.clin_mut_enc = nn.Sequential(
            nn.Linear(clin_mut_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, latent_dim_clin),
            nn.ReLU()
        )

        self.expr_enc = expr_autoencoder

        fusion_dim = latent_dim_clin + latent_dim_expr

        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

    def forward(self, x_clin, x_expr):
        z_clin = self.clin_mut_enc(x_clin)
        z_expr = self.expr_enc(x_expr)[1]

        z = torch.cat([z_clin, z_expr], dim=1)
        risk = self.fusion(z).squeeze(-1)

        return risk

In [171]:
# ============================================================
# AUTOENCODER TRAINING (PER FOLD)
# ============================================================
def train_survival_autoencoder(X_expr, y_time, y_event,
                               latent_dim=64, epochs=200):

    input_dim = X_expr.shape[1]
    ae = SurvivalAwareAutoencoder(input_dim, latent_dim)

    optimizer = optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-4)
    recon_fn = nn.MSELoss()

    X_expr = X_expr.float()
    y_time = y_time.float()
    y_event = y_event.float()

    best_loss = float("inf")
    best_weights = copy.deepcopy(ae.state_dict())

    for epoch in range(epochs):
        ae.train()
        optimizer.zero_grad()

        recon, z, risk = ae(X_expr)

        recon_loss = recon_fn(recon, X_expr)
        surv_loss = cox_loss(risk, y_time, y_event)

        loss = recon_loss + 0.05 * surv_loss

        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_weights = copy.deepcopy(ae.state_dict())

    ae.load_state_dict(best_weights)
    return ae

In [172]:
# ============================================================
# TRAIN DUAL ENCODER
# ============================================================
def train_dual_encoder_survival_net(X_clin, X_expr, y_time, y_event,
                                   expr_autoencoder,
                                   latent_dim_clin=32,
                                   latent_dim_expr=64,
                                   epochs=500):

    model = DualEncoderSurvivalNet(
        clin_mut_dim=X_clin.shape[1],
        expr_autoencoder=expr_autoencoder,
        latent_dim_clin=latent_dim_clin, 
        latent_dim_expr=latent_dim_expr
    )

    X_clin = X_clin.float()
    X_expr = X_expr.float()
    y_time = y_time.float()
    y_event = y_event.float()

    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

    best_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        risk = model(X_clin, X_expr)
        loss = cox_loss(risk, y_time, y_event)

        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    return model


In [173]:
# ============================================================
# C-INDEX
# ============================================================
def compute_cindex(model, times, events, X_clin, X_expr):
    model.eval()
    with torch.no_grad():
        risk = model(X_clin, X_expr).cpu().numpy()

    return concordance_index(times, -risk, events)


In [182]:
def run_pipeline(clinical_df, mutation_df, X_expr, y_time, y_event, k=100, latent_dim_clin=32, latent_dim_expr=64):

    # ============================================================
    # 1. HOLDOUT SPLIT (NO LEAKAGE)
    # ============================================================
    idx = np.arange(len(y_event))

    train_idx, test_idx = train_test_split(
        idx,
        test_size=0.2,
        stratify=y_event,
        random_state=42
    )

    clinical_train, clinical_test = clinical_df.iloc[train_idx], clinical_df.iloc[test_idx]
    mutation_train, mutation_test = mutation_df.iloc[train_idx], mutation_df.iloc[test_idx]

    expr_train, expr_test = X_expr.iloc[train_idx], X_expr.iloc[test_idx]

    y_time_train, y_time_test = y_time[train_idx], y_time[test_idx]
    y_event_train, y_event_test = y_event[train_idx], y_event[test_idx]

    # ============================================================
    # 2. CV SETUP
    # ============================================================
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_scores = []

    for fold, (tr, va) in enumerate(skf.split(clinical_train, y_event_train)):

        print(f"\n================ FOLD {fold+1} ================")

        # -------------------------
        # Split fold data
        # -------------------------
        clin_tr = clinical_train.iloc[tr]
        clin_va = clinical_train.iloc[va]

        mut_tr = mutation_train.iloc[tr]
        mut_va = mutation_train.iloc[va]

        expr_tr = expr_train.iloc[tr]
        expr_va = expr_train.iloc[va]

        yt_tr = y_time_train[tr]
        yt_va = y_time_train[va]

        ye_tr = y_event_train[tr]
        ye_va = y_event_train[va]

        # ==================================================
        # SURVIVAL-BASED GENE FILTERING
        # ==================================================

        gene_idx, selected_genes, scores = select_survival_genes_fast(
            expr_tr,
            yt_tr,
            ye_tr,
            top_k=k
        )

        expr_tr = expr_tr.iloc[:, gene_idx]
        expr_va = expr_va.iloc[:, gene_idx]

        print(f"Selected {len(gene_idx)} genes")
        print("Top genes:")
        print(selected_genes[:10])

        # ============================================================
        # 3. BUILD X_CLIN (LEAKAGE SAFE: FIT INSIDE FOLD)
        # ============================================================
        builder = ClinicalMutationBuilder()

        builder.fit(clin_tr, mut_tr)

        Xc_tr = builder.transform(clin_tr, mut_tr)
        Xc_va = builder.transform(clin_va, mut_va)

        # ============================================================
        # 4. FEATURE CLEANING (IMPORTANT)
        # ============================================================
        Xc_tr, keep = remove_constant_features(Xc_tr)
        Xc_va = Xc_va[:, keep]

        # ============================================================
        # 5. PREPROCESS EXPRESSIONS (FOLD SAFE)
        # ============================================================
        pre = SurvivalPreprocessor()
        pre.fit(Xc_tr, expr_tr)

        Xc_tr, Xe_tr = pre.transform(Xc_tr, expr_tr)
        Xc_va, Xe_va = pre.transform(Xc_va, expr_va)

        # ============================================================
        # 6. TENSORS
        # ============================================================
        Xc_tr = torch.tensor(Xc_tr).float()
        Xe_tr = torch.tensor(Xe_tr).float()
        yt_tr = torch.tensor(yt_tr).float()
        ye_tr = torch.tensor(ye_tr).float()

        Xc_va = torch.tensor(Xc_va).float()
        Xe_va = torch.tensor(Xe_va).float()

        # ============================================================
        # 7. AUTOENCODER (PER FOLD)
        # ============================================================
        ae = train_survival_autoencoder(Xe_tr, yt_tr, ye_tr, latent_dim=latent_dim_expr)

        # ============================================================
        # 8. DUAL ENCODER MODEL
        # ============================================================
        model = train_dual_encoder_survival_net(
            Xc_tr, Xe_tr, yt_tr, ye_tr, ae, 
            latent_dim_clin=latent_dim_clin, 
            latent_dim_expr=latent_dim_expr
        )

        # ============================================================
        # 9. EVALUATION
        # ============================================================
        cindex = compute_cindex(model, yt_va, ye_va, Xc_va, Xe_va)

        cv_scores.append(cindex)

        print("C-index:", cindex)

    # ============================================================
    # 10. FINAL TRAINING (FULL TRAIN SET)
    # ============================================================
    print("\n================ FINAL TRAINING ================")

    builder = ClinicalMutationBuilder()
    builder.fit(clinical_train, mutation_train)

    Xc_train = builder.transform(clinical_train, mutation_train)
    Xc_test = builder.transform(clinical_test, mutation_test)

    Xc_train, keep = remove_constant_features(Xc_train)
    Xc_test = Xc_test[:, keep]

    # Gene Filtering
    gene_idx, selected_genes, scores = select_survival_genes_fast(
        expr_train,
        y_time_train,
        y_event_train,
        top_k=k
    )
    expr_train = expr_train.iloc[:, gene_idx]
    expr_test = expr_test.iloc[:, gene_idx]

    print("Top Genes:")
    print(selected_genes[:10])
    print("Final selected genes:", len(gene_idx))
    
    pre = SurvivalPreprocessor()
    pre.fit(Xc_train, expr_train)

    Xc_train, Xe_train = pre.transform(Xc_train, expr_train)
    Xc_test, Xe_test = pre.transform(Xc_test, expr_test)

    Xc_train = torch.tensor(Xc_train).float()
    Xe_train = torch.tensor(Xe_train).float()
    yt_train = torch.tensor(y_time_train).float()
    ye_train = torch.tensor(y_event_train).float()

    Xc_test = torch.tensor(Xc_test).float()
    Xe_test = torch.tensor(Xe_test).float()

    ae = train_survival_autoencoder(Xe_train, yt_train, ye_train, latent_dim=latent_dim_expr)

    final_model = train_dual_encoder_survival_net(
        Xc_train, Xe_train, yt_train, ye_train, ae,
        latent_dim_expr=latent_dim_expr,
        latent_dim_clin=latent_dim_clin
    )

    test_cindex = compute_cindex(final_model, y_time_test, y_event_test,
                                 Xc_test, Xe_test)

    print("\nCV mean:", np.mean(cv_scores))
    print("CV std:", np.std(cv_scores))
    print("TEST C-index:", test_cindex)

    return final_model, cv_scores, test_cindex

### Load Data and Preprocessing

In [180]:
# Load clinical metadata
clinical = pd.read_csv(here.here("data", "processed", "clinical_data.csv"))
# Load expression data
expression_data = pd.read_csv(here.here("data", "processed", "expression_data.csv"))
# Mutation binary data
mutation_binary = pd.read_csv(here.here("data", "processed", "mutation_binary_data.csv"))
# Mutation classified data
mutation_classified = pd.read_csv(here.here("data", "processed", "mutation_classified_data.csv"))

pd.set_option('display.max_columns', None)

print("Data shapes:")
print(f"  Clinical: {clinical.shape}")
print(f"  Expression: {expression_data.shape}")
print(f"  Mutation Binary: {mutation_binary.shape}")
print(f"  Mutation Classified: {mutation_classified.shape}")

Data shapes:
  Clinical: (1904, 31)
  Expression: (1904, 489)
  Mutation Binary: (1904, 173)
  Mutation Classified: (1904, 173)


## Preprocess Clinical Features

In [176]:
# Prepare clinical data
df = clinical.copy()
time_col = "overall_survival_months"
event_col = "death_from_cancer"

# Define outcome variables
df["event"] = (df[event_col] == "Died of Disease").astype(int)
df["time"] = df[time_col]

print(f"Event distribution:\n{df['event'].value_counts()}")

# Group rare cancer types into "Other" category
rare_threshold = 50

value_counts = df["cancer_type_detailed"].value_counts()
rare_classes = value_counts[value_counts < rare_threshold].index
df["cancer_type_detailed_clean"] = df["cancer_type_detailed"].replace(rare_classes, "Other")

# Do same for histologic subtype
value_counts = df["tumor_other_histologic_subtype"].value_counts()
rare_classes = value_counts[value_counts < rare_threshold].index
df["tumor_other_histologic_subtype_clean"] = df["tumor_other_histologic_subtype"].replace(rare_classes, "Other")

# Drop identifiers, outcome columns, and pre-modified columns
drop_cols = [
    "patient_id", "cancer_type", "cohort", "cancer_type_detailed",
    "er_status_measured_by_ihc", "her2_status_measured_by_snp6",
    "tumor_other_histologic_subtype", "integrative_cluster", "oncotree_code",
    "radio_therapy", "chemotherapy", "type_of_breast_surgery", "hormone_therapy",
    "overall_survival", "nottingham_prognostic_index", "3-gene_classifier_subtype",
    "tumor_other_histologic_subtype_clean", time_col, event_col
]

X_clinical = df.drop(columns=drop_cols + ["event", "time"])
y_time = df["time"].values
y_event = df["event"].values

# Identify feature types
numeric_cols = X_clinical.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_clinical.select_dtypes(include=["object", "category"]).columns.tolist()

# Fill missing categorical data
X_clinical[categorical_cols] = X_clinical[categorical_cols].fillna("Unknown")

print(f"\nClinical features:")
print(f"  Numeric: {len(numeric_cols)}")
print(f"  Categorical: {len(categorical_cols)}")

Event distribution:
event
0    1282
1     622
Name: count, dtype: int64

Clinical features:
  Numeric: 5
  Categorical: 9


## Preprocess Mutation Features

In [177]:
# Option A: TMB only (1 feature) — simplest possible mutation signal
tmb_only = mutation_binary.sum(axis=1).values.astype(np.float32).reshape(-1, 1)

# Option B: Top mutated genes only — keep genes mutated in >5% of patients
mutation_freq = mutation_binary.mean(axis=0)
frequent_genes = mutation_freq[mutation_freq > 0.05].index
print(f"Genes mutated in >5% of patients: {len(frequent_genes)}")
print(frequent_genes.tolist())

# mut_frequent = mutation_binary[frequent_genes].values.astype(np.float32)
mut_frequent = mutation_binary[frequent_genes].astype(np.float32)

mut_frequent["TMB"] = tmb_only

print(f"Frequent mutation + TMB matrix shape: {mut_frequent.shape}")

Genes mutated in >5% of patients: 25
['pik3ca_mut', 'tp53_mut', 'muc16_mut', 'ahnak2_mut', 'kmt2c_mut', 'syne1_mut', 'gata3_mut', 'map3k1_mut', 'ahnak_mut', 'dnah11_mut', 'cdh1_mut', 'dnah2_mut', 'kmt2d_mut', 'ush2a_mut', 'ryr2_mut', 'dnah5_mut', 'herc2_mut', 'pde4dip_mut', 'akap9_mut', 'tg_mut', 'birc6_mut', 'utrn_mut', 'tbx3_mut', 'col6a3_mut', 'arid1a_mut']
Frequent mutation + TMB matrix shape: (1904, 26)


In [178]:
# Final check of shapes of input data
print(X_clinical.shape)
print(mutation_binary.shape)
print(mut_frequent.shape)
print(expression_data.shape)
print(X_clinical.shape)
print(y_time.shape)
print(y_event.shape)

(1904, 14)
(1904, 173)
(1904, 26)
(1904, 489)
(1904, 14)
(1904,)
(1904,)


# Run with full mutation dataset, and top genes = 100

In [159]:
final_model, cv_scores, test_cindex = run_pipeline(X_clinical,mutation_binary,expression_data,y_time,y_event)


================ FOLD 1 ================
Selected 100 genes
Top genes:
['aurka' 'bcl2' 'gsk3b' 'e2f2' 'e2f7' 'fancd2' 'eif4ebp1' 'stat5b' 'mapt'
 'chek1']
[Feature Filter]
Original: 211 features
Kept: 209 features
Dropped: 2 constant features
C-index: 0.6898650306748466

================ FOLD 2 ================
Selected 100 genes
Top genes:
['aurka' 'gsk3b' 'bcl2' 'e2f2' 'fancd2' 'ccne1' 'chek1' 'cdc25a' 'slc19a1'
 'e2f7']
[Feature Filter]
Original: 211 features
Kept: 209 features
Dropped: 2 constant features
C-index: 0.591630455318345

================ FOLD 3 ================
Selected 100 genes
Top genes:
['aurka' 'bcl2' 'stat5b' 'mapt' 'gsk3b' 'e2f2' 'chek1' 'ccne1' 'cdk1'
 'e2f8']
[Feature Filter]
Original: 211 features
Kept: 209 features
Dropped: 2 constant features
C-index: 0.6225988700564972

================ FOLD 4 ================
Selected 100 genes
Top genes:
['aurka' 'gsk3b' 'e2f2' 'stat5b' 'fancd2' 'e2f7' 'rps6kb2' 'bcl2' 'stat5a'
 'cdc25a']
[Feature Filter]
Original: 211 f

# Run with only top mutations, with TMB added, and top genes = 100

In [161]:
final_model, cv_scores, test_cindex = run_pipeline(X_clinical,mut_frequent,expression_data,y_time,y_event)


================ FOLD 1 ================
Selected 100 genes
Top genes:
['aurka' 'bcl2' 'gsk3b' 'e2f2' 'e2f7' 'fancd2' 'eif4ebp1' 'stat5b' 'mapt'
 'chek1']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6687116564417178

================ FOLD 2 ================
Selected 100 genes
Top genes:
['aurka' 'gsk3b' 'bcl2' 'e2f2' 'fancd2' 'ccne1' 'chek1' 'cdc25a' 'slc19a1'
 'e2f7']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6013602871717363

================ FOLD 3 ================
Selected 100 genes
Top genes:
['aurka' 'bcl2' 'stat5b' 'mapt' 'gsk3b' 'e2f2' 'chek1' 'ccne1' 'cdk1'
 'e2f8']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6034389584868582

================ FOLD 4 ================
Selected 100 genes
Top genes:
['aurka' 'gsk3b' 'e2f2' 'stat5b' 'fancd2' 'e2f7' 'rps6kb2' 'bcl2' 'stat5a'
 'cdc25a']
[Feature Filter]
Original: 64 feature

## Run with different values for K

In [164]:
for k in [25, 50, 200]:
    print(f"\n\nTOP {k} GENES USED:\n")
    final_model, cv_scores, test_cindex = run_pipeline(X_clinical,mut_frequent,expression_data,y_time,y_event, k=k)



TOP 25 GENES USED:


================ FOLD 1 ================
Selected 25 genes
Top genes:
['aurka' 'bcl2' 'gsk3b' 'e2f2' 'e2f7' 'fancd2' 'eif4ebp1' 'stat5b' 'mapt'
 'chek1']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6836319018404908

================ FOLD 2 ================
Selected 25 genes
Top genes:
['aurka' 'gsk3b' 'bcl2' 'e2f2' 'fancd2' 'ccne1' 'chek1' 'cdc25a' 'slc19a1'
 'e2f7']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6552994521065558

================ FOLD 3 ================
Selected 25 genes
Top genes:
['aurka' 'bcl2' 'stat5b' 'mapt' 'gsk3b' 'e2f2' 'chek1' 'ccne1' 'cdk1'
 'e2f8']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6348808646524196

================ FOLD 4 ================
Selected 25 genes
Top genes:
['aurka' 'gsk3b' 'e2f2' 'stat5b' 'fancd2' 'e2f7' 'rps6kb2' 'bcl2' 'stat5a'
 'cdc25a']
[Feature Filter]
Or

In [165]:
final_model, cv_scores, test_cindex = run_pipeline(X_clinical,mut_frequent,expression_data,y_time,y_event, k=10)


================ FOLD 1 ================
Selected 10 genes
Top genes:
['aurka' 'bcl2' 'gsk3b' 'e2f2' 'e2f7' 'fancd2' 'eif4ebp1' 'stat5b' 'mapt'
 'chek1']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.7059141104294478

================ FOLD 2 ================
Selected 10 genes
Top genes:
['aurka' 'gsk3b' 'bcl2' 'e2f2' 'fancd2' 'ccne1' 'chek1' 'cdc25a' 'slc19a1'
 'e2f7']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6339977328547137

================ FOLD 3 ================
Selected 10 genes
Top genes:
['aurka' 'bcl2' 'stat5b' 'mapt' 'gsk3b' 'e2f2' 'chek1' 'ccne1' 'cdk1'
 'e2f8']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
C-index: 0.6185703758290346

================ FOLD 4 ================
Selected 10 genes
Top genes:
['aurka' 'gsk3b' 'e2f2' 'stat5b' 'fancd2' 'e2f7' 'rps6kb2' 'bcl2' 'stat5a'
 'cdc25a']
[Feature Filter]
Original: 64 features
Ke

In [ ]:
for k in [10, 15, 25, 100,150, 200]:
    for l in [8,16,32,64]:
        print(f"\n\nSELECTED GENES: {k}\nAE LAT DIMS: {l}\n")
        final_model, cv_scores, test_cindex = run_pipeline(X_clinical,mut_frequent,expression_data,y_time,y_event, k=k, latent_dim_expr=l)



SELECTED GENES: 10
AE LAT DIMS: 8


================ FOLD 1 ================
Selected 10 genes
Top genes:
['aurka' 'gsk3b' 'e2f2' 'bcl2' 'fancd2' 'e2f7' 'stat5b' 'cdc25a' 'chek1'
 'mapt']
[Feature Filter]
Original: 64 features
Kept: 64 features
Dropped: 0 constant features
